# Reproducing Andrews, Geng, Voitiuk et al. (2024) from dandiset 001132

**Paper.** Andrews JP, Geng J, Voitiuk K, et al. (2024). Multimodal evaluation of network activity
and optogenetic interventions in human hippocampal slices. *Nature Neuroscience* 27, 2487-2499.
[doi:10.1038/s41593-024-01782-5](https://doi.org/10.1038/s41593-024-01782-5).

**Dataset.** [DANDI:001132](https://dandiarchive.org/dandiset/001132/0.241130.1903), version
`0.241130.1903` ([doi:10.48324/dandi.001132/0.241130.1903](https://doi.org/10.48324/dandi.001132/0.241130.1903)),
streamed from the archive. The paper names the dandiset without a version. There are two published
versions, 0.241119.1801 and 0.241130.1903; the data check below shows that they hold the same 69
assets (same asset identifiers and sizes), so the most recent one is pinned.

**Claims.** `claims/andrews-2024.yaml`, written from the NIH author manuscript in PubMed Central
(PMC11881784) with its Methods, legends and supplement, and checked by a second reader.

**What the dandiset holds.** Each of the 69 files holds one HD-MEA recording as raw voltage only:
no sorted units, no light (GPIO) events, no stimulation log, no condition or medium labels and no
session description. Almost every claim rests on the paper's curated units and on which recording
and which trials make each condition, so they are blocked. The notebook surveys every recording,
recovers what it can of the light events from the voltage, and tests the one claim about the cohort.

| Claim | Status |
|---|---|
| table1-cohort | tested |
| units-per-recording | blocked: data_missing (no sorted units) |
| transduction-range, reporter-gcl-pyramidal | blocked: data_missing (histology) |
| patch-hckcr1-hyperpolarizing | blocked: data_missing (patch clamp) |
| simulation-dampening | blocked: not_runnable (NEST 3.4 and NESTML 5 cannot be installed together in a locked environment) |
| fig3a-spontaneous-reduction, physio-bic-reduction-all-slices, physio-bic-amplitude-unchanged, table2-fr-decrease-physio-bic, table2-fr-decrease-0mg, zero-mg-reduction, ka-light-reduction, chr2-increase, closed-loop-bicuculline, intensity-threshold, s3-reduction-by-condition | blocked: data_missing (units; light events also missing outside 3C) |
| zero-mg-isolated-bursts, ka-rhythmic-bursts, ka-bursts-during-light, ka-theta-coherence, fig4-firing-propagation, fig4-theta-propagation | blocked: data_missing (units; conditions not labelled) |
| transduction-not-explaining-response | blocked: data_missing (depends on the Table 2 claims and on histology) |
| waveform-clusters, waveform-gcl-association, opto-response-clusters, opto-waveform-association, opto-cluster3-profile | blocked: data_missing (units) |

A variant on multi-unit threshold crossings for slice 3C, the only slice whose light events can be
recovered, is shown below; it decides no outcome.

In [ ]:
import resource
import time
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import requests
from scipy import stats
from statsmodels.stats.multitest import multipletests

import hdmea

pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 200)
T0 = time.time()
print("dandiset", hdmea.DANDISET, "version", hdmea.VERSION)

## Data check

The files and versions, then the metadata of every file. Each check of the metadata against the values is stated under its output.

In [ ]:
versions = requests.get(f"{hdmea.API}/dandisets/{hdmea.DANDISET}/versions/", timeout=60).json()["results"]
print("versions:", [(v["version"], v["status"], v["asset_count"], v["size"]) for v in versions])
published = [v["version"] for v in versions if v["version"] != "draft"]
listing = {v: {(a["asset_id"], a["path"], a["size"]) for a in hdmea.assets(v)} for v in published}
print("published versions hold the same assets:", len({frozenset(s) for s in listing.values()}) == 1)
meta = requests.get(f"{hdmea.API}/dandisets/{hdmea.DANDISET}/versions/{hdmea.VERSION}/", timeout=60).json()
summ = meta["assetsSummary"]
print("assetsSummary: files", summ["numberOfFiles"], "subjects", summ["numberOfSubjects"],
      "bytes", summ["numberOfBytes"], "variables", summ["variableMeasured"])

In [ ]:
ASSETS = hdmea.assets()
def load_meta(a):
    url = hdmea.s3_url(a["asset_id"])
    return dict(recording=hdmea.recording_name(a["path"]), path=a["path"], size_gb=a["size"] / 1e9, url=url,
                **hdmea.metadata(url))
with ThreadPoolExecutor(8) as ex:
    M = list(ex.map(load_meta, ASSETS))
info = pd.DataFrame([{k: v for k, v in m.items() if k not in ("x", "y", "electrode")} for m in M])
info["slice"] = info.recording.str.split("-").str[0]
info["patient"] = info.slice.str[-1]
info["duration_s"] = info.n_samples / info.rate
order = info.slice.str[:-1].astype(int) * 100 + info.recording.str.split("-").str[1].astype(int)
info = info.assign(o=order).sort_values("o").drop(columns="o").reset_index(drop=True)
print(info[["recording", "subject_id", "age", "sex", "species", "session_start", "n_channels", "duration_s",
            "size_gb"]].to_string())

In [ ]:
print("files:", len(info), " slices:", info.slice.nunique(), " patients:", info.patient.nunique())
print("recordings per slice:", info.groupby("slice", sort=False).size().to_dict())
print("subject_id equals the recording name in every file:", (info.subject_id == info.recording).all())
print("age and sex constant within each patient:",
      info.groupby("patient").apply(lambda g: g[["age", "sex"]].drop_duplicates().shape[0] == 1, include_groups=False).all())
print("channels constant within each slice:", info.groupby("slice").n_channels.nunique().eq(1).all())
print("dtype", info.dtype.unique(), " sampling rate (Hz)", info.rate.unique(), " conversion (V/count)", info.conversion.unique(),
      " offset", info.offset.unique(), " unit", info.unit.unique())
print("chunks", info.chunks.astype(str).unique(), " filters", info.filters.astype(str).unique())
print("duration (s): min %.1f, median %.1f, max %.1f, total %.0f" % (info.duration_s.min(), info.duration_s.median(),
      info.duration_s.max(), info.duration_s.sum()))
print("session_start values:", sorted(info.session_start.str[:10].unique()))
print("session_description non-empty:", (info.session_description != "").sum())
print("electrode locations:", sorted({l for m in M for l in m["location"]}), " device:", info.device.unique())
print("files with a units table:", info.has_units.sum(), " with intervals:", info.intervals.sum(),
      " with processing modules:", sum(len(p) > 0 for p in info.processing),
      " with stimulus series:", sum(len(s) > 0 for s in info.stimulus),
      " acquisition objects per file:", info.n_acquisition.unique())
print("objects outside the ElectricalSeries and general/:", sorted({p for ps in info.paths for p in ps}))
xs = np.concatenate([m["x"] for m in M]); ys = np.concatenate([m["y"] for m in M]); el = np.concatenate([m["electrode"] for m in M])
print("electrode x (um): %.1f to %.1f; y: %.1f to %.1f; all on a 17.5 um grid: %s; electrode ids %d to %d" % (
      xs.min(), xs.max(), ys.min(), ys.max(), bool(np.allclose(np.r_[xs, ys] / 17.5, np.round(np.r_[xs, ys] / 17.5))),
      el.min(), el.max()))

The files are named by slice (`1A` to `12G`, the slice IDs of Table 1) and a recording index; the
letter is the patient. Each file's `subject_id` is the recording (`3C-0`), so the archive counts
69 subjects where there are 7 patients and 12 slices. Every `session_start_time` is 2024-09-28, the
conversion date, not the recording date (the paper's code names recordings from 2023). The
sampling rate (20 kHz), the channel counts (up to 1,024) and the electrode positions on the
17.5 µm grid of a 26,400-electrode MaxWell array are consistent with the Methods. Nothing in the
files says which medium, drug or light protocol a recording belongs to, which opsin a slice
carries, or which of the paper's recordings a file is.

## Survey of every recording

One pass over the first 64-channel group of every file gives the channel-mean voltage in 10 ms
bins (to find light steps) and multi-unit threshold crossings per channel in 100 ms bins (helper
`hdmea.scan`; parameters and their sources are in its docstring). It reads about 12 GB and takes
about 20 minutes. It was timed on two recordings while preparing (11 s for a 190 s recording,
1.3 GB peak memory).

In [ ]:
t = time.time()
SCAN = {}
for m in M:
    SCAN[m["recording"]] = hdmea.scan(m["url"])
print("scanned %d recordings in %.1f min; %.1f GB compressed read; peak memory %.2f GB" % (
      len(SCAN), (time.time() - t) / 60, sum(s["compressed"] for s in SCAN.values()) / 1e9,
      resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6))
rng = pd.DataFrame([{"slice": r.split("-")[0], "vmin": s["vmin"], "vmax": s["vmax"],
                     "rail_frac_max": float(s["rail"].max()), "rail_frac_median": float(np.median(s["rail"]))}
                    for r, s in SCAN.items()])
rng = rng.groupby("slice", sort=False).agg(vmin=("vmin", "min"), vmax=("vmax", "max"),
                                           rail_frac_max=("rail_frac_max", "max"), rail_frac_median=("rail_frac_median", "max"))
rng["conversion_uV"] = info.groupby("slice").conversion.agg(lambda c: sorted(set(np.round(c * 1e6, 3)))).reindex(rng.index)
print("values (counts) of the first channel group per slice, the largest fraction of its channels at 0 or 1023 in any")
print("100 ms bin, and the conversion to volts:")
print(rng.round(3).to_string())

### Recordings deposited twice

The compressed bytes of the first and the last chunk of the first channel group, compared across files; files of equal length are listed too.

In [ ]:
dig = pd.DataFrame([{"recording": r, "first": s["digest_first"], "last": s["digest_last"], "n": s["n"]} for r, s in SCAN.items()])
print("files sharing a first chunk:", dig[dig.duplicated("first", keep=False)].recording.tolist())
print("files sharing a last chunk:", dig[dig.duplicated("last", keep=False)].recording.tolist())
same_len = dig[dig.duplicated("n", keep=False)].groupby("n").recording.apply(list)
print("files of identical length (samples):", same_len.tolist())

No two files share a chunk, so no recording is deposited twice; files of equal length differ in their data.

### Light events

The Methods say the light ON and OFF times were stored in the MaxWell file's GPIO bit and in a
stimulation log (Methods, Design of optogenetic system). Neither is in the NWB files. In slice 3C
the LED leaves a step in the voltage of every channel at onset and the opposite step at offset;
these steps are counted here at several thresholds against the bin-to-bin noise.

In [ ]:
rows = []
for r, s in SCAN.items():
    row = {"recording": r, "duration_s": round(s["n"] / hdmea.FS, 1)}
    for th in (20, 40, 60, 100):
        L = hdmea.light_steps(s["dc"], th)
        row[f"trials@{th}"] = len(L["on"])
        row[f"up/down@{th}"] = f"{L['n_up']}/{L['n_down']}"
    row["noise"] = round(L["noise"], 2)
    row["max_step"] = round(float(np.abs(np.diff(s["dc"])).max()), 1)
    rows.append(row)
steps = pd.DataFrame(rows)
print(steps.to_string())
LIGHT = {r: hdmea.light_steps(SCAN[r]["dc"]) for r in steps.recording}
with_light = [r for r, L in LIGHT.items() if len(L["on"])]
print("\nrecordings with light trials at the main threshold (%g counts):" % hdmea.STEP_MIN, with_light)
for r in with_light:
    L = LIGHT[r]
    print(r, "onsets (s):", np.round(L["on"], 2).tolist(), " ON durations (s):", np.round(L["off"] - L["on"], 2).tolist(),
          " onset steps:", np.round(L["up"]).astype(int).tolist(), " offset steps:", np.round(L["down"]).astype(int).tolist())

Paired light steps (onset +67 to +168 counts, offset -67 to -88 counts, in 10 ms bins against a
bin-to-bin noise of 0.5 to 1.6 counts) occur in the eight 3C recordings, and in no other file at
the main threshold of 60 counts: outside 3C, one file (2B-1, noise 2.5) has a single unpaired upward
jump of 73 counts, and the largest jump in any other file is 57 counts (10F-7), unpaired. At 20 and
40 counts the extra steps outside 3C are also unpaired. No other slice, including the ChR2 slice 4C
and the HcKCR1 slices the paper illuminated in 10-trial protocols (9E to 12G) or in closed loop
(5C, 7D, 8D), shows a light step in this channel group. Light events can therefore be recovered for
slice 3C only. The 3C light steps are in the voltage itself, so they share its clock by
construction; there is no trial table to compare with.

Each light period lasts 10.0 s (3C-0: 5.0 s). 3C-6 and 3C-7 hold six light periods each, about 20 s
apart (onset to onset); in 3C-6 the first starts 1 s into the file, so 5 of its 6 have a full 10 s
pre window. 3C-1 to 3C-5 hold one 10 s period each, starting 10 s into a 40 to 70 s file, which fits
the intensity sweeps of Extended Data Fig. 7, but the files do not say so. The claims file has five
trials for 3C (Fig. 3a, Extended Data Fig. 4 legend); 3C-6 has five with a full pre window.

## table1-cohort

> Recordings came from 12 hippocampal slices from 7 adult patients with refractory temporal lobe
> epilepsy (5 female, 2 male, aged 20 to 52), 8 slices transduced with HcKCR1, 2 with ChR2 and 2
> with no opsin.

Criterion: reproduced when the dandiset's subjects and sessions resolve to 12 slices from 7
patients with the opsins, ages and sexes of Table 1; partially reproduced when the HcKCR1 slices
of Table 2 (3C, 5C, 7D, 8D, 9E, 10F, 11G, 12G) are all present and identifiable but others are not.
Table 1 is read from the paper (`paper/`); its ages and sexes per patient are A 20 M, B 44 F,
C 40 F, D 46 F, E 46 F, F 52 F, G 35 M.

In [ ]:
pat = info.groupby("patient").agg(age=("age", "first"), sex=("sex", "first"), slices=("slice", lambda s: sorted(set(s))),
                                  recordings=("recording", "size"))
pat["age_years"] = pat.age.str.extract(r"P(\d+)Y").astype(int)
print(pat.to_string())
print("\nslices:", info.slice.nunique(), " patients:", info.patient.nunique())
print("female patients:", (pat.sex == "F").sum(), " male:", (pat.sex == "M").sum(),
      " ages %d to %d" % (pat.age_years.min(), pat.age_years.max()))
hckcr1 = ["3C", "5C", "7D", "8D", "9E", "10F", "11G", "12G"]
print("HcKCR1 slices of Table 2 present by name:", all(s in set(info.slice) for s in hckcr1))
opsin_fields = [c for c in ("virus", "opsin", "genotype", "strain") if any(c in p for ps in info.paths for p in ps)]
print("opsin, virus or genotype recorded anywhere in the files:", bool(opsin_fields),
      "; in the dandiset metadata:", any(w in str(meta).lower() for w in ("hckcr1", "chr2", "aav9")))

**Result: partially reproduced.** The 69 files resolve to 12 slices named as in Table 1 (1A to
12G), from 7 patients (A to G), 5 female and 2 male, aged 20 to 52, with each patient's age and sex
as in Table 1; the 8 HcKCR1 slices of Table 2 are present by name. The opsin of each slice is not
recorded in the files or the dandiset's metadata, so the count of 8 HcKCR1, 2 ChR2 and 2 no-opsin
slices can be had only by reading the slice names against the paper's Table 1, not from the
dandiset.

## Claims that need sorted units or light events

`units-per-recording` and every claim on firing rates, amplitudes, bursts, propagation, waveform
or opto-response clusters rests on the paper's curated units (Kilosort2, then manual and automatic
curation). The files hold no units table, so these claims are blocked as `data_missing`. A re-sort
with Kilosort2 needs MATLAB and a CUDA GPU, which the locked environment does not have, and the
manual curation cannot be repeated. The light-trial claims also need the light events, which are
recoverable from the voltage only in slice 3C (above), and the claims on 0-mg, 0-mg+KA and
bicuculline recordings need to know which file is which condition, which no file says.

### Variant: the 3C light trials in the raw voltage

This is not the paper's method and decides no outcome. It shows what the dataset holds around the
only light trials it lets one find. Per 100 ms bin of the first 64-channel group, it gives the
fraction of channels at 0 or 1023 (the ends of the range the values take) and the multi-unit
threshold crossings (300 to 6000 Hz, 5 MAD per channel and chunk). Crossings are counted in the
claims file's windows, pre (10 s before onset), ON (10 s from onset) and OFF (onset + 10 to
+ 20 s), averaged over the recording's trials, and compared by two-tailed paired t-tests over
channels with crossings, Holm-adjusted over the three comparisons, as the claims file's opening
comment does for units. A channel is not a unit: one channel can carry several units and one
unit can reach several channels.

In [ ]:
def compare(rec, rates):
    keep = (rates["pre"] + rates["on"] + rates["off"]) > 0
    pre, on, off = rates["pre"][keep], rates["on"][keep], rates["off"][keep]
    p = [stats.ttest_rel(a, b).pvalue for a, b in ((pre, on), (on, off), (pre, off))]
    padj = multipletests(p, method="holm")[1]
    return {"recording": rec, "trials": rates["n_trials"], "channels": int(keep.sum()),
            "pre_Hz": pre.mean(), "on_Hz": on.mean(), "off_Hz": off.mean(),
            "change_on_vs_pre_%": 100 * (on.mean() - pre.mean()) / pre.mean(),
            "p_pre_on": padj[0], "p_on_off": padj[1], "p_pre_off": padj[2],
            "channels_lower_on": int((on < pre).sum())}

def saturation(rec):
    rail, on = SCAN[rec]["rail"], LIGHT[rec]["on"]
    out = {"pre": [], "on": [], "off": [], "on_first_0.5s": []}
    for t in on:
        b = int(round(t * 10))
        if b - 100 < 0 or b + 200 > len(rail):
            continue
        out["pre"].append(rail[b - 100:b].mean()); out["on"].append(rail[b:b + 100].mean())
        out["off"].append(rail[b + 100:b + 200].mean()); out["on_first_0.5s"].append(rail[b:b + 5].mean())
    return {f"rail_{k}": (np.mean(v) if v else np.nan) for k, v in out.items()}

res = []
for r in with_light:
    w = hdmea.window_rates(SCAN[r]["counts"], LIGHT[r]["on"])
    row = compare(r, w) if w["n_trials"] else {"recording": r, "trials": 0}
    res.append({**row, **saturation(r)})
var = pd.DataFrame(res)
print(var.round(4).to_string())

In [ ]:
# Crossings per second (summed over the 64 channels) and the fraction of channels at the rails,
# through the six light periods of 3C-7
r = "3C-7"
c = SCAN[r]["counts"].sum(1)
per_s = c[: len(c) // 10 * 10].reshape(-1, 10).sum(1)
rail_s = SCAN[r]["rail"][: len(SCAN[r]["rail"]) // 10 * 10].reshape(-1, 10).max(1)
L = LIGHT[r]
state = np.full(len(per_s), "pre/off", dtype=object)
for a, b in zip(L["on"], L["off"]):
    state[int(np.floor(a)):int(np.ceil(b))] = "ON"
tab = pd.DataFrame({"crossings_per_s": per_s[: len(rail_s)], "max_rail_frac": rail_s, "state": state[: len(rail_s)]})
print(tab.groupby("state").agg(median=("crossings_per_s", "median"), min=("crossings_per_s", "min"),
                               max=("crossings_per_s", "max"), seconds_over_1000=("crossings_per_s", lambda v: int((v > 1000).sum())),
                               max_rail=("max_rail_frac", "max")).to_string())
first = int(np.floor(L["on"][0]))
print("seconds %d to %d around the first onset (%.2f s):" % (first - 3, first + 25, L["on"][0]))
print(tab.iloc[first - 3:first + 25].T.to_string())

**Result: blocked (data_missing); variant only.** fig3a-spontaneous-reduction needs the curated
units of 3C, which are not in the dandiset. In the raw voltage, light onset drives 80 to 94% of the
channels to the ends of their range in the first 0.5 s (6% stay there through the light in 3C-7),
and seconds with over 1,000 crossings occur both during light (28 s in 3C-7) and outside it (8 s),
next to the light steps. In the five single-trial files 3C-1 to 3C-5 crossings fall by 86 to 90%
from pre to ON (Holm-adjusted p 0.0001 or less); in the multi-trial files 3C-6 (5 trials) and 3C-7
(6 trials) they do not (-16% and +6%, adjusted p 0.82 and 0.87), with the pre windows following
earlier light periods. Saturation and these transients confound the crossings, so the variant
cannot say how light changes firing.

## Summary

| Claim | Outcome |
|---|---|
| table1-cohort | partially reproduced |
| units-per-recording | blocked: data_missing |
| transduction-range | blocked: data_missing |
| reporter-gcl-pyramidal | blocked: data_missing |
| patch-hckcr1-hyperpolarizing | blocked: data_missing |
| simulation-dampening | blocked: not_runnable |
| fig3a-spontaneous-reduction | blocked: data_missing |
| physio-bic-reduction-all-slices | blocked: data_missing |
| physio-bic-amplitude-unchanged | blocked: data_missing |
| table2-fr-decrease-physio-bic | blocked: data_missing |
| table2-fr-decrease-0mg | blocked: data_missing |
| zero-mg-reduction | blocked: data_missing |
| zero-mg-isolated-bursts | blocked: data_missing |
| ka-rhythmic-bursts | blocked: data_missing |
| ka-light-reduction | blocked: data_missing |
| ka-bursts-during-light | blocked: data_missing |
| ka-theta-coherence | blocked: data_missing |
| fig4-firing-propagation | blocked: data_missing |
| fig4-theta-propagation | blocked: data_missing |
| closed-loop-bicuculline | blocked: data_missing |
| intensity-threshold | blocked: data_missing |
| chr2-increase | blocked: data_missing |
| transduction-not-explaining-response | blocked: data_missing |
| s3-reduction-by-condition | blocked: data_missing |
| waveform-clusters | blocked: data_missing |
| waveform-gcl-association | blocked: data_missing |
| opto-response-clusters | blocked: data_missing |
| opto-waveform-association | blocked: data_missing |
| opto-cluster3-profile | blocked: data_missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** Only the cohort claim is tested, and it is tested
on the file names and subject metadata. The variant on 3C uses threshold crossings on 64 channels
in place of curated units, and is confounded by amplifier saturation at light onset.

**What was found about the dataset.** The 69 files hold raw HD-MEA voltage only: no sorted units,
no light events (the GPIO bit and the stimulation log the Methods describe), no condition, medium
or opsin labels, no session description, and no link from a file to the paper's recording names.
Light periods can be recovered from a voltage step in the 3C recordings only, where light onset
drives most channels to the ends of their range. In at least one 5C recording, 34% of the first channel
group sits at the ends of the range in the median 100 ms bin. `subject_id` is the
recording, so the archive counts 69 subjects for 7 patients, and every session start time is the
conversion date. No recording is deposited twice.

In [ ]:
print("notebook ran in %.1f min; peak memory %.2f GB" % ((time.time() - T0) / 60,
      resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6))